# EDA cơ bản cho tập dữ liệu `hm_v1` trước embedding

Notebook này kiểm chứng các số liệu và nhận định ở mục 2.2 của luận văn. Chỉ dùng `items.parquet` và bốn bảng giao dịch `train.parquet`, `rerank_train.parquet`, `valid.parquet`, `test.parquet` trong `data/hm`. Không dùng tập giao dịch nguồn lớn, embedding, mô hình hay kết quả xếp hạng.

**Đơn vị đếm:** một dòng trong bảng giao dịch là một lượt mua sản phẩm. Một khách có thể mua nhiều sản phẩm trong một ngày. Các cửa sổ sau train chỉ gồm khách có mua trong chính cửa sổ đó; mẫu gốc có 50.000 khách đủ điều kiện từ giai đoạn train.

## 1. Nạp và kiểm tra cấu trúc đầu vào

Chạy từ gốc dự án hoặc từ thư mục notebook. Nếu dữ liệu nằm ở chỗ khác, đặt biến môi trường `HM_DATA_DIR`.

In [1]:
from pathlib import Path
import os
import polars as pl

candidates = []
if os.getenv("HM_DATA_DIR"):
    candidates.append(Path(os.environ["HM_DATA_DIR"]))
candidates += [Path.cwd() / "data" / "hm", Path.cwd().parent / "data" / "hm", Path.cwd().parent.parent / "data" / "hm"]
data_dir = next((p.resolve() for p in candidates if (p / "train.parquet").is_file()), None)
if data_dir is None:
    raise FileNotFoundError("Không tìm thấy data/hm/train.parquet; hãy đặt HM_DATA_DIR.")

names = ("train", "rerank_train", "valid", "test")
required = {"customer_id", "article_id", "t_dat", "price", "sales_channel_id", "split"}
frames = {}
for name in names:
    path = data_dir / f"{name}.parquet"
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pl.read_parquet(path)
    missing = required - set(frame.columns)
    assert not missing, f"{name}: thiếu cột {missing}"
    assert frame["split"].unique().to_list() == [name], f"{name}: nhãn split không khớp"
    frames[name] = frame
items = pl.read_parquet(data_dir / "items.parquet")
assert "item_id" in items.columns
print("Thư mục dữ liệu:", data_dir)
print("Schema giao dịch:", frames["train"].schema)
print("Schema sản phẩm:", items.schema)

Thư mục dữ liệu: D:\WorkSpace\Work\DATN\data\hm
Schema giao dịch: Schema([('customer_id', String), ('article_id', String), ('t_dat', Date), ('price', Float64), ('sales_channel_id', Int64), ('split', String)])
Schema sản phẩm: Schema([('item_id', String), ('prod_name', String), ('product_type_name', String), ('product_group_name', String), ('graphical_appearance_name', String), ('colour_group_name', String), ('department_name', String), ('section_name', String), ('detail_desc', String)])


## 2. Mẫu dữ liệu thực tế (Dataset Samples)

In mẫu các bản ghi thực tế từ các bảng giao dịch (`train`, `test`) và danh mục sản phẩm (`items`), cùng một ví dụ ghép nối (join) chuỗi giao dịch của một khách hàng với thông tin sản phẩm để trực quan hóa luồng dữ liệu tương tác.

In [2]:
# Mẫu các bản ghi giao dịch trong tập train và test
print("--- BẢNG GIAO DỊCH TRAIN ---")
print(frames["train"].head(5))

print("\n---BẢNG GIAO DỊCH TEST ---")
print(frames["test"].head(3))

--- BẢNG GIAO DỊCH TRAIN ---
shape: (5, 6)
┌─────────────────────────────────┬────────────┬────────────┬──────────┬──────────────────┬───────┐
│ customer_id                     ┆ article_id ┆ t_dat      ┆ price    ┆ sales_channel_id ┆ split │
│ ---                             ┆ ---        ┆ ---        ┆ ---      ┆ ---              ┆ ---   │
│ str                             ┆ str        ┆ date       ┆ f64      ┆ i64              ┆ str   │
╞═════════════════════════════════╪════════════╪════════════╪══════════╪══════════════════╪═══════╡
│ 0001420326c217a367048472e05395… ┆ 0566940004 ┆ 2018-11-06 ┆ 0.011847 ┆ 2                ┆ train │
│ 0001420326c217a367048472e05395… ┆ 0566941001 ┆ 2018-11-06 ┆ 0.013542 ┆ 2                ┆ train │
│ 0001420326c217a367048472e05395… ┆ 0651242003 ┆ 2018-11-06 ┆ 0.016932 ┆ 2                ┆ train │
│ 0001420326c217a367048472e05395… ┆ 0396135041 ┆ 2018-11-07 ┆ 0.006763 ┆ 2                ┆ train │
│ 0001420326c217a367048472e05395… ┆ 0615797001 ┆ 2018-11-

In [3]:
# Mẫu bảng danh mục sản phẩm và nội dung mô tả chi tiết
print("--- MẪU 5 DÒNG ĐẦU BẢNG DANH MỤC SẢN PHẨM (ITEMS) ---")
print(items.head(5))

print("\n--- VÍ DỤ NỘI DUNG MÔ TẢ CHI TIẾT (detail_desc) CỦA 3 SẢN PHẨM MẪU ---")
sample_descs = items.filter(pl.col("detail_desc").is_not_null()).head(3)
for row in sample_descs.iter_rows(named=True):
    print(f"[{row['item_id']}] {row['prod_name']} ({row['product_type_name']} | {row['colour_group_name']}):")
    print(f"   \"{row['detail_desc']}\"\n")

--- MẪU 5 DÒNG ĐẦU BẢNG DANH MỤC SẢN PHẨM (ITEMS) ---
shape: (5, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ prod_name ┆ product_t ┆ product_g ┆ … ┆ colour_gr ┆ departmen ┆ section_n ┆ detail_d │
│ ---       ┆ ---       ┆ ype_name  ┆ roup_name ┆   ┆ oup_name  ┆ t_name    ┆ ame       ┆ esc      │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 010877501 ┆ Strap top ┆ Vest top  ┆ Garment   ┆ … ┆ Black     ┆ Jersey    ┆ Womens    ┆ Jersey   │
│ 5         ┆           ┆           ┆ Upper     ┆   ┆           ┆ Basic     ┆ Everyday  ┆ top with │
│           ┆           ┆           ┆ body      ┆   ┆           ┆           ┆ Basics    ┆ narrow   │
│           ┆          

In [4]:
# Mẫu chuỗi giao dịch của một khách hàng ghép nối với danh mục sản phẩm
sample_cust = frames["train"]["customer_id"][0]
cust_history = (
    frames["train"]
    .filter(pl.col("customer_id") == sample_cust)
    .head(5)
    .join(
        items.select(["item_id", "prod_name", "product_type_name", "colour_group_name"]),
        left_on="article_id",
        right_on="item_id",
        how="left",
    )
    .select([
        "customer_id",
        "t_dat",
        "article_id",
        "prod_name",
        "product_type_name",
        "colour_group_name",
        "price",
        "sales_channel_id",
    ])
)
print(f"Mẫu 5 giao dịch đầu của khách {sample_cust[:16]}...:")
print(cust_history)

Mẫu 5 giao dịch đầu của khách 0001420326c217a3...:
shape: (5, 8)
┌────────────┬────────────┬────────────┬────────────┬───────────┬───────────┬──────────┬───────────┐
│ customer_i ┆ t_dat      ┆ article_id ┆ prod_name  ┆ product_t ┆ colour_gr ┆ price    ┆ sales_cha │
│ d          ┆ ---        ┆ ---        ┆ ---        ┆ ype_name  ┆ oup_name  ┆ ---      ┆ nnel_id   │
│ ---        ┆ date       ┆ str        ┆ str        ┆ ---       ┆ ---       ┆ f64      ┆ ---       │
│ str        ┆            ┆            ┆            ┆ str       ┆ str       ┆          ┆ i64       │
╞════════════╪════════════╪════════════╪════════════╪═══════════╪═══════════╪══════════╪═══════════╡
│ 0001420326 ┆ 2018-11-06 ┆ 0566940004 ┆ Greg solid ┆ Swimwear  ┆ Dark Blue ┆ 0.011847 ┆ 2         │
│ c217a36704 ┆            ┆            ┆            ┆ bottom    ┆           ┆          ┆           │
│ 8472e05395 ┆            ┆            ┆            ┆           ┆           ┆          ┆           │
│ …          ┆            

## 3. Thống kê cơ bản và kiểm tra chất lượng dữ liệu

Kiểm tra tỷ lệ giá trị khuyết (missing/null), phân phối giá giao dịch (`price`), tỷ lệ kênh bán hàng (`sales_channel_id`), và các đặc trưng phân loại nổi bật trong danh mục hàng hóa.

In [5]:
# Kiểm tra giá trị khuyết thiếu (Null / Missing values)
print("--- GIÁ TRỊ KHUYẾT THIẾU BẢNG GIAO DỊCH TRAIN ---")
print(frames["train"].null_count())

print("\n--- GIÁ TRỊ KHUYẾT THIẾU BẢNG DANH MỤC SẢN PHẨM (ITEMS) ---")
print(items.null_count())

# Khẳng định: toàn bộ bảng giao dịch không có giá trị khuyết
assert frames["train"].null_count().sum_horizontal()[0] == 0, "Bảng train không được có giá trị null" 

--- GIÁ TRỊ KHUYẾT THIẾU BẢNG GIAO DỊCH TRAIN ---
shape: (1, 6)
┌─────────────┬────────────┬───────┬───────┬──────────────────┬───────┐
│ customer_id ┆ article_id ┆ t_dat ┆ price ┆ sales_channel_id ┆ split │
│ ---         ┆ ---        ┆ ---   ┆ ---   ┆ ---              ┆ ---   │
│ u32         ┆ u32        ┆ u32   ┆ u32   ┆ u32              ┆ u32   │
╞═════════════╪════════════╪═══════╪═══════╪══════════════════╪═══════╡
│ 0           ┆ 0          ┆ 0     ┆ 0     ┆ 0                ┆ 0     │
└─────────────┴────────────┴───────┴───────┴──────────────────┴───────┘

--- GIÁ TRỊ KHUYẾT THIẾU BẢNG DANH MỤC SẢN PHẨM (ITEMS) ---
shape: (1, 9)
┌─────────┬───────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ item_id ┆ prod_name ┆ product_ty ┆ product_g ┆ … ┆ colour_gr ┆ departmen ┆ section_n ┆ detail_de │
│ ---     ┆ ---       ┆ pe_name    ┆ roup_name ┆   ┆ oup_name  ┆ t_name    ┆ ame       ┆ sc        │
│ u32     ┆ u32       ┆ ---        ┆ ---       ┆   ┆ -

### 3.1. Kiểm tra null, trùng lặp và khóa liên kết trên toàn bộ hm_v1

Phân biệt **dòng trùng hoàn toàn** (mọi cột giống nhau) với **mua cùng sản phẩm trong cùng ngày** (cùng khách, sản phẩm, ngày; có thể khác giá hoặc kênh). Dữ liệu không có mã giao dịch hay số lượng, nên không thể khẳng định các dòng này là lỗi nhập liệu. Notebook đo cả hai loại trước khi quyết định cách dùng: giữ nguyên các dòng giao dịch để bảo toàn số lượt mua; chỉ gộp sản phẩm trùng trong cùng giỏ hàng theo ngày khi dựng chuỗi đầu vào. Các lần mua lại ở ngày khác được giữ trong lịch sử.

Kiểm tra thêm khóa nối danh mục, giá không hợp lệ, kênh bán ngoài miền giá trị và null/chuỗi rỗng của metadata. Mọi phép đếm đều chạy trên bốn phần train, rerank_train, valid, test của tập đang dùng.

In [6]:
catalog_ids = items.select(pl.col("item_id").alias("article_id"))
audit_rows = []
for name, frame in frames.items():
    exact_unique = frame.unique().height
    day_unique = frame.unique(subset=["customer_id", "article_id", "t_dat"]).height
    audit_rows.append({
        "phần": name,
        "dòng": frame.height,
        "ô_null": sum(frame.null_count().row(0)),
        "trùng_hoàn_toàn": frame.height - exact_unique,
        "trùng_khách_SP_ngày": frame.height - day_unique,
        "SP_không_có_trong_catalog": frame.join(catalog_ids, on="article_id", how="anti").height,
        "giá_không_hợp_lệ": frame.filter(pl.col("price").is_null() | ~pl.col("price").is_finite() | (pl.col("price") <= 0)).height,
        "kênh_ngoài_1_2": frame.filter(~pl.col("sales_channel_id").is_in([1, 2])).height,
    })
audit = pl.DataFrame(audit_rows)
print("KIỂM TRA CẢ BỐN PHẦN GIAO DỊCH")
print(audit)
print(audit.to_dicts())
assert audit["ô_null"].sum() == 0
assert audit["SP_không_có_trong_catalog"].sum() == 0

metadata_rows = []
for column in items.columns:
    field = items[column]
    metadata_rows.append({
        "cột": column,
        "null": field.null_count(),
        "chuỗi_rỗng_hoặc_chỉ_khoảng_trắng": field.str.strip_chars().eq("").sum() if field.dtype == pl.String else 0,
    })
metadata_audit = pl.DataFrame(metadata_rows)
print("\nKIỂM TRA METADATA SẢN PHẨM")
print(metadata_audit)
print("item_id trùng:", items.height - items["item_id"].n_unique())
assert items.height == items["item_id"].n_unique()

train_unique_day = frames["train"].unique(subset=["customer_id", "article_id", "t_dat"])
repeat_pairs = (
    train_unique_day.group_by("customer_id", "article_id")
    .agg(pl.col("t_dat").n_unique().alias("số_ngày"))
    .filter(pl.col("số_ngày") > 1)
    .height
)
print("\nCặp khách–sản phẩm được mua ở từ hai ngày khác nhau trong train:", repeat_pairs)
print("Quy tắc chuẩn bị: giữ mọi dòng giao dịch; khi dựng giỏ theo ngày, mỗi sản phẩm chỉ xuất hiện một lần trong giỏ.")
desc_ready = items.select(pl.col("detail_desc").fill_null("").alias("detail_desc"))
print("detail_desc sau fill_null:", desc_ready["detail_desc"].null_count())
assert desc_ready["detail_desc"].null_count() == 0 and desc_ready.height == items.height


KIỂM TRA CẢ BỐN PHẦN GIAO DỊCH
shape: (4, 8)
┌─────────────┬─────────┬────────┬─────────────┬────────────┬────────────┬────────────┬────────────┐
│ phần        ┆ dòng    ┆ ô_null ┆ trùng_hoàn_ ┆ trùng_khác ┆ SP_không_c ┆ giá_không_ ┆ kênh_ngoài │
│ ---         ┆ ---     ┆ ---    ┆ toàn        ┆ h_SP_ngày  ┆ ó_trong_ca ┆ hợp_lệ     ┆ _1_2       │
│ str         ┆ i64     ┆ i64    ┆ ---         ┆ ---        ┆ talog      ┆ ---        ┆ ---        │
│             ┆         ┆        ┆ i64         ┆ i64        ┆ ---        ┆ i64        ┆ i64        │
│             ┆         ┆        ┆             ┆            ┆ i64        ┆            ┆            │
╞═════════════╪═════════╪════════╪═════════════╪════════════╪════════════╪════════════╪════════════╡
│ train       ┆ 1242735 ┆ 0      ┆ 112643      ┆ 121790     ┆ 0          ┆ 0          ┆ 0          │
│ rerank_trai ┆ 10452   ┆ 0      ┆ 874         ┆ 948        ┆ 0          ┆ 0          ┆ 0          │
│ n           ┆         ┆        ┆            


Cặp khách–sản phẩm được mua ở từ hai ngày khác nhau trong train: 41326
Quy tắc chuẩn bị: giữ mọi dòng giao dịch; khi dựng giỏ theo ngày, mỗi sản phẩm chỉ xuất hiện một lần trong giỏ.
detail_desc sau fill_null: 0


In [7]:
# Thống kê phân phối giá (price) và kênh bán hàng (sales_channel_id)
print("--- PHÂN PHỐI GIÁ GIAO DỊCH (PRICE) TRONG TẬP HUẤN LUYỆN ---")
print(frames["train"].select("price").describe())

print("\n--- PHÂN BỐ KÊNH BÁN HÀNG (SALES_CHANNEL_ID) ---")
channel_dist = (
    frames["train"]["sales_channel_id"]
    .value_counts(sort=True)
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(2).alias("tỉ_lệ_%")
    )
)
print(channel_dist)

--- PHÂN PHỐI GIÁ GIAO DỊCH (PRICE) TRONG TẬP HUẤN LUYỆN ---
shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ price      │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 1.242735e6 │
│ null_count ┆ 0.0        │
│ mean       ┆ 0.027345   │
│ std        ┆ 0.018727   │
│ min        ┆ 0.000153   │
│ 25%        ┆ 0.015237   │
│ 50%        ┆ 0.025407   │
│ 75%        ┆ 0.033881   │
│ max        ┆ 0.50678    │
└────────────┴────────────┘

--- PHÂN BỐ KÊNH BÁN HÀNG (SALES_CHANNEL_ID) ---
shape: (2, 3)
┌──────────────────┬────────┬─────────┐
│ sales_channel_id ┆ count  ┆ tỉ_lệ_% │
│ ---              ┆ ---    ┆ ---     │
│ i64              ┆ u32    ┆ f64     │
╞══════════════════╪════════╪═════════╡
│ 2                ┆ 850626 ┆ 68.45   │
│ 1                ┆ 392109 ┆ 31.55   │
└──────────────────┴────────┴─────────┘


In [8]:
# Thống kê cơ bản các đặc trưng phân loại của danh mục sản phẩm
print("--- TOP 5 NHÓM SẢN PHẨM (product_group_name) ---")
print(items["product_group_name"].value_counts(sort=True).head(5))

print("\n--- TOP 5 LOẠI SẢN PHẨM (product_type_name) ---")
print(items["product_type_name"].value_counts(sort=True).head(5))

print("\n--- TOP 5 NHÓM MÀU SẮC (colour_group_name) ---")
print(items["colour_group_name"].value_counts(sort=True).head(5))

print("\n--- THỐNG KÊ ĐỘ DÀI VĂN BẢN MÔ TẢ (detail_desc) THEO SỐ KÝ TỰ ---")
desc_lens = items.filter(pl.col("detail_desc").is_not_null()).select(
    pl.col("detail_desc").str.len_chars().alias("độ_dài_ký_tự")
)
print(desc_lens.describe())

--- TOP 5 NHÓM SẢN PHẨM (product_group_name) ---
shape: (5, 2)
┌────────────────────┬───────┐
│ product_group_name ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ Garment Upper body ┆ 42741 │
│ Garment Lower body ┆ 19812 │
│ Garment Full body  ┆ 13292 │
│ Accessories        ┆ 11158 │
│ Underwear          ┆ 5490  │
└────────────────────┴───────┘

--- TOP 5 LOẠI SẢN PHẨM (product_type_name) ---
shape: (5, 2)
┌───────────────────┬───────┐
│ product_type_name ┆ count │
│ ---               ┆ ---   │
│ str               ┆ u32   │
╞═══════════════════╪═══════╡
│ Trousers          ┆ 11169 │
│ Dress             ┆ 10362 │
│ Sweater           ┆ 9302  │
│ T-shirt           ┆ 7904  │
│ Top               ┆ 4155  │
└───────────────────┴───────┘

--- TOP 5 NHÓM MÀU SẮC (colour_group_name) ---
shape: (5, 2)
┌───────────────────┬───────┐
│ colour_group_name ┆ count │
│ ---               ┆ ---   │
│ str               ┆ u32   │
╞═══════════════════╪

In [9]:
import json

# Kiểm tra tệp cấu hình thực nghiệm và manifest gói ảnh thô nếu có
config_path = data_dir / "run_config.json"
if config_path.is_file():
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    print("Cấu hình tập thực nghiệm (run_config.json):")
    for k, v in cfg.items():
        print(f"  - {k}: {v}")

img_manifest_path = data_dir / "Image" / "images_manifest.json"
if img_manifest_path.is_file():
    with open(img_manifest_path, "r", encoding="utf-8") as f:
        img_meta = json.load(f)
    print("\nThông tin gói ảnh sản phẩm (images_manifest.json):")
    print(f"  - Tổng số ảnh: {img_meta.get('total_images'):,}")
    print(f"  - Dung lượng nén: {img_meta.get('total_size_mb', 0):.1f} MB")
    print(f"  - Chế độ lọc: {img_meta.get('filter_mode')}")
    assert img_meta.get("total_images") == 72_811

Cấu hình tập thực nghiệm (run_config.json):
  - dataset: H&M Personalized Fashion Recommendations
  - version: hm_v1
  - profile: quick
  - seed: 20260922
  - customers: 50000
  - min_train_events: 3
  - max_events_per_customer: 100

Thông tin gói ảnh sản phẩm (images_manifest.json):
  - Tổng số ảnh: 72,811
  - Dung lượng nén: 1198.0 MB
  - Chế độ lọc: active_only


## 4. Quy mô và khoảng thời gian của bốn phần dữ liệu

Đếm trực tiếp trên các dòng Parquet đang dùng; số khách và sản phẩm là số mã khác nhau **trong từng phần**, không cộng các cột này giữa các phần.

In [10]:
summary = pl.DataFrame([
    {
        "phần": name,
        "lượt_mua": frames[name].height,
        "khách": frames[name]["customer_id"].n_unique(),
        "sản_phẩm": frames[name]["article_id"].n_unique(),
        "ngày_đầu": str(frames[name]["t_dat"].min()),
        "ngày_cuối": str(frames[name]["t_dat"].max()),
    }
    for name in names
])
print(summary)
print("Tổng lượt mua trong bốn phần:", sum(frames[name].height for name in names))
print("Số dòng danh mục:", items.height)
assert frames["train"]["customer_id"].n_unique() == 50_000
assert sum(frames[name].height for name in names) == 1_273_677
assert items.height == 105_542

shape: (4, 6)
┌──────────────┬──────────┬───────┬──────────┬────────────┬────────────┐
│ phần         ┆ lượt_mua ┆ khách ┆ sản_phẩm ┆ ngày_đầu   ┆ ngày_cuối  │
│ ---          ┆ ---      ┆ ---   ┆ ---      ┆ ---        ┆ ---        │
│ str          ┆ i64      ┆ i64   ┆ i64      ┆ str        ┆ str        │
╞══════════════╪══════════╪═══════╪══════════╪════════════╪════════════╡
│ train        ┆ 1242735  ┆ 50000 ┆ 71357    ┆ 2018-09-20 ┆ 2020-09-01 │
│ rerank_train ┆ 10452    ┆ 3063  ┆ 4569     ┆ 2020-09-02 ┆ 2020-09-08 │
│ valid        ┆ 10690    ┆ 3030  ┆ 4505     ┆ 2020-09-09 ┆ 2020-09-15 │
│ test         ┆ 9800     ┆ 2799  ┆ 4200     ┆ 2020-09-16 ┆ 2020-09-22 │
└──────────────┴──────────┴───────┴──────────┴────────────┴────────────┘


Tổng lượt mua trong bốn phần: 1273677
Số dòng danh mục: 105542


## 5. Số lượt mua của mỗi khách trong lịch sử huấn luyện

Tính trên `train.parquet`: nhóm theo `customer_id`, đếm số dòng từng khách, rồi lấy trung bình và trung vị của **50.000** số đếm đó. Đây là số lượt mua trong lịch sử đã giữ lại, không phải tổng mua suốt đời của khách trong dữ liệu nguồn.

In [11]:
train = frames["train"]
purchases_per_customer = train.group_by("customer_id").agg(pl.len().alias("lượt_mua"))
mean_purchases = purchases_per_customer["lượt_mua"].mean()
median_purchases = purchases_per_customer["lượt_mua"].median()
print("Số khách:", purchases_per_customer.height)
print(f"Trung bình lượt mua/khách: {mean_purchases:.4f}")
print(f"Trung vị lượt mua/khách: {median_purchases:g}")
print("Nhỏ nhất / lớn nhất:", purchases_per_customer["lượt_mua"].min(), "/", purchases_per_customer["lượt_mua"].max())
assert purchases_per_customer.height == 50_000
assert round(mean_purchases, 2) == 24.85
assert median_purchases == 14

Số khách: 50000
Trung bình lượt mua/khách: 24.8547
Trung vị lượt mua/khách: 14
Nhỏ nhất / lớn nhất: 3 / 100


## 6. Bao nhiêu khách phát sinh giao dịch trong từng cửa sổ?

Lấy 50.000 khách ở phần train làm mẫu số. Mỗi bảng tuần sau chỉ có các khách phát sinh ít nhất một lượt mua trong tuần tương ứng. Phép trừ `50.000 − số khách có mua` cho biết bao nhiêu khách không có giao dịch trong cửa sổ; không suy diễn rằng họ rời hệ thống.

In [12]:
base_users = train["customer_id"].n_unique()
participation = pl.DataFrame([
    {
        "cửa_sổ": name,
        "khách_có_mua": frames[name]["customer_id"].n_unique(),
        "khách_không_mua": base_users - frames[name]["customer_id"].n_unique(),
        "tỉ_lệ_có_mua_%": round(100 * frames[name]["customer_id"].n_unique() / base_users, 3),
    }
    for name in names[1:]
])
print(participation)
assert all(participation["khách_có_mua"] < base_users)
assert participation["khách_có_mua"].to_list() == [3063, 3030, 2799]

shape: (3, 4)
┌──────────────┬──────────────┬─────────────────┬────────────────┐
│ cửa_sổ       ┆ khách_có_mua ┆ khách_không_mua ┆ tỉ_lệ_có_mua_% │
│ ---          ┆ ---          ┆ ---             ┆ ---            │
│ str          ┆ i64          ┆ i64             ┆ f64            │
╞══════════════╪══════════════╪═════════════════╪════════════════╡
│ rerank_train ┆ 3063         ┆ 46937           ┆ 6.126          │
│ valid        ┆ 3030         ┆ 46970           ┆ 6.06           │
│ test         ┆ 2799         ┆ 47201           ┆ 5.598          │
└──────────────┴──────────────┴─────────────────┴────────────────┘


## 7. Khách kiểm thử có lịch sử huấn luyện không?

So sánh **tập mã khách khác nhau** của `test.parquet` với tập mã khách của `train.parquet`. Nếu phép `anti join` rỗng, mọi khách trong test đã có ít nhất một giao dịch ở train. Kết luận này chỉ đúng với mẫu `hm_v1`; notebook không đo chất lượng gợi ý cho khách mới.

In [13]:
train_users = train.select("customer_id").unique()
test_users = frames["test"].select("customer_id").unique()
test_without_train = test_users.join(train_users, on="customer_id", how="anti")
print("Khách kiểm thử:", test_users.height)
print("Khách kiểm thử không có lịch sử train:", test_without_train.height)
print(f"Tỉ lệ khách kiểm thử có lịch sử train: {(1 - test_without_train.height / test_users.height) * 100:.1f}%")
assert test_users.height == 2799
assert test_without_train.height == 0

Khách kiểm thử:

 2799
Khách kiểm thử không có lịch sử train: 0
Tỉ lệ khách kiểm thử có lịch sử train: 100.0%


## 8. Kiểm tra bổ sung về danh mục

Phân biệt quy mô danh mục với số sản phẩm thực sự được mua trong train; kiểm tra độ thiếu của trường mô tả văn bản. Các con số này hỗ trợ phần mô tả bảng sản phẩm trong mục 2.2.

In [14]:
train_items = train["article_id"].n_unique()
missing_desc = items["detail_desc"].is_null().sum()
print("Sản phẩm trong danh mục:", items.height)
print("Sản phẩm có giao dịch trong train:", train_items)
print("Sản phẩm danh mục chưa xuất hiện trong train:", items.height - train_items)
print(f"detail_desc thiếu: {missing_desc}/{items.height} = {missing_desc / items.height * 100:.2f}%")
assert train_items == 71_357
assert items["item_id"].n_unique() == items.height

Sản phẩm trong danh mục: 105542
Sản phẩm có giao dịch trong train: 71357
Sản phẩm danh mục chưa xuất hiện trong train: 34185
detail_desc thiếu: 416/105542 = 0.39%


## 9. Đối chiếu với câu viết trong luận văn

| Nhận định ở mục 2.2 | Bằng chứng trong notebook |
|---|---|
| Cấu trúc dữ liệu giao dịch và danh mục sản phẩm | Mục 2: Mẫu dữ liệu `train`, `test`, `items` và mẫu ghép nối giao dịch |
| Không có giá trị thiếu ở các trường phân loại, `detail_desc` thiếu ~0,39% | Mục 3 & 8: Kiểm tra `null_count` trên các bảng và đếm 416 dòng thiếu |
| Gói ảnh đang sử dụng gồm 72.811 ảnh cho các sản phẩm có hoạt động | Mục 3: Đối chiếu `images_manifest.json` trong thư mục `Image` |
| Mỗi khách có trung bình 24,85 và trung vị 14 lượt mua trong lịch sử huấn luyện | Mục 5: nhóm `train` theo `customer_id`, tính mean và median |
| Số khách có mua trong từng tuần sau nhỏ hơn 50.000 | Mục 6: lần lượt 3.063, 3.030 và 2.799 khách |
| Tất cả khách test đã có lịch sử train | Mục 7: anti join có 0 khách |
| Không thể đánh giá khách hoàn toàn mới trên mẫu này | Suy luận từ điều kiện chọn khách và kết quả anti join ở mục 7 |

**Giới hạn:** Trung bình/trung vị ở đây dựa trên lịch sử train đã cắt tối đa 100 lượt mua cũ mỗi khách. Khẳng định về khách mới là giới hạn của tập thực nghiệm này, không phải kết luận về mọi khách hàng H&M.